## WSW Gates - Beach Courses

Generate beach courses for Weymouth Speed Week

In [1]:
import os
import sys

import pyproj
import jinja2

### Constants

Coordinates of South West Coast Path waypoints were identified on Google Earth

In [2]:
# SE waypoint
SE_WAYPOINT = (-2.46500000, 50.57482500)

# NW waypoint
NW_WAYPOINT = (-2.46826944, 50.57890556)

In [3]:
# The true length of the course
TRACK_LENGTH = 500

# Course of the KML allows for GPS inaccuracy / buoys being slightly mislaid
COURSE_LENGTH = TRACK_LENGTH + 20

# Course is slightly shifted towards Chesil
COURSE_SHIFT = 10

# Course extends well into the harbour for foils
COURSE_WIDTH = 1500

# Azimuth is roughly parallel to the road
COURSE_AZIMUTH = 330

In [4]:
# Courses directory relative to project
COURSES_DIR = 'courses'

# GTX and KML directories within courses
GTX_DIR = 'gtx'
KML_DIR = 'kml'

### Common Functions for GTX and KML

Calculation for corners, start + finish, etc

In [5]:
def calculateCorners(mid_lon, mid_lat, azimuth, distance, shift, gate_width):
    '''Calculate corners for course'''

    # c1 = start line (shore), c3 = finish line (shore)
    c1 = geod.fwd(mid_lon, mid_lat, (azimuth + 180) % 360, distance / 2)
    c3 = geod.fwd(mid_lon, mid_lat, azimuth, distance / 2)
    
    # Move c1 + c3 towards chesil by 25 meters
    c1 = geod.fwd(c1[0], c1[1], (azimuth - 90) % 360, shift)
    c3 = geod.fwd(c3[0], c3[1], (azimuth - 90) % 360, shift)

    # c2 = start line (harbour), c4 = finish line (harbour)
    c2 = geod.fwd(c1[0], c1[1], (azimuth + 90) % 360, gate_width)
    c4 = geod.fwd(c3[0], c3[1], (azimuth + 90) % 360, gate_width)

    return c1, c2, c3, c4


def calculateStartFinish(c1, c2, c3, c4):
    '''Calculate start and finish points for course'''

    # Simple average will be correct to within a few centimeters
    start = ((c1[0] + c2[0]) / 2, (c1[1] + c2[1]) / 2)
    finish = ((c3[0] + c4[0]) / 2, (c3[1] + c4[1]) / 2)

    return start, finish

### Jinja Setup

Prepare environment for Jinja templates

In [6]:
def getJinjaPaths():
    '''Prepare Jinja environment'''
    
    projdir = os.path.realpath(os.path.join(sys.path[0], '..'))
    
    coursesPath = os.path.join(projdir, COURSES_DIR)
    gtxPath = os.path.join(coursesPath, GTX_DIR)
    kmlPath = os.path.join(coursesPath, KML_DIR)

    return gtxPath, kmlPath


def getJinjaEnv(gtxPath, kmlPath):
    '''Get Jinja environment for GTX and KML'''
    
    loader = jinja2.FileSystemLoader([gtxPath, kmlPath])
    env = jinja2.Environment(loader=loader, autoescape=True, trim_blocks=True, lstrip_blocks=True)

    return env

### Generate Gate File

Use Jinja to generate .gtx file from template

In [7]:
def saveGtx(fn, jinjaEnv, mid_lon, mid_lat):
    '''Save GTX for GPSResults'''
    
    # c1 + c2 = start line, c3 + c4 = finish line
    c1, c2, c3, c4 = calculateCorners(mid_lon, mid_lat, COURSE_AZIMUTH, TRACK_LENGTH, COURSE_SHIFT, COURSE_WIDTH)

    # Start and finish points are at the middle of each line
    start, finish = calculateStartFinish(c1, c2, c3, c4)
    
    # Corners of the course
    corners_lat_lon = "{:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f}".format(
        c1[1], c1[0], c2[1], c2[0], c3[1], c3[0], c4[1], c4[0])
    
    # Prepare Gate XML
    template = jinjaEnv.get_template("template.gtx")
    gtxData = template.render(track_length=-TRACK_LENGTH, gate_width=COURSE_WIDTH,
                          start_lon=round(start[0], 7), start_lat=round(start[1], 7),
                          finish_lon=round(finish[0], 7), finish_lat=round(finish[1], 7),
                          corners_lat_lon=corners_lat_lon)
    
    # Save Gate XML
    with open(fn, 'w', encoding='utf-8') as f:
    	f.write(gtxData)

### Generate KML File

Use Jinja to generate .kml file from template

In [8]:
def saveKml(fn, jinjaEnv, mid_lon, mid_lat, name):
    '''Save KML for Google Earth'''
    
    # c1 + c2 = start line, c3 + c4 = finish line
    c1, c2, c3, c4 = calculateCorners(mid_lon, mid_lat, COURSE_AZIMUTH, COURSE_LENGTH, COURSE_SHIFT, COURSE_WIDTH)

    # Start and finish points are at the middle of each line
    start, finish = calculateStartFinish(c1, c2, c3, c4)
    
    # Polygon coordinates must join up the start and end
    polygon_coordinates = \
        "{:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0".format(
        c1[0], c1[1], c3[0], c3[1], c4[0], c4[1], c2[0], c2[1], c1[0], c1[1])
    
    # Prepare KML
    template = jinjaEnv.get_template("template.kml")
    kml = template.render(name=name,
                          start_lon=round(start[0], 7), start_lat=round(start[1], 7),
                          finish_lon=round(finish[0], 7), finish_lat=round(finish[1], 7),
                          polygon_coordinates=polygon_coordinates)
    
    # Save KML
    with open(fn, 'w', encoding='utf-8') as f:
    	f.write(kml)

### Calculate distance and azimuth between SWCP waypoints

The pyproj library returns distances in metres, and azimuths betwen -180 and +180

In [9]:
geod = pyproj.Geod(ellps='WGS84')

In [10]:
# Use SWCP waypoints
lon1, lat1 = SE_WAYPOINT
lon2, lat2 = NW_WAYPOINT

# Determine forward and back azimuths, plus distance between the waypoints
forward_azimuth, back_azimuth, distance = geod.inv(lon1, lat1, lon2, lat2)

# Convert negative values to positive values
forward_azimuth = (forward_azimuth + 360) % 360
back_azimuth = (back_azimuth + 360) % 360

# Report the results
print(f"Distance: {distance:.3f} meters")
print(f"Heading: {forward_azimuth:.3f} degrees")

Distance: 509.587 meters
Heading: 332.971 degrees


### Generate Multiple Courses

Typically 50 meter intervals

In [11]:
interval = 50

w1 = geod.fwd_intermediate(lon1, lat1, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)
w2 = geod.fwd_intermediate(lon2, lat2, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)

In [12]:
i = 3

mid_lon = (w1.lons[i] + w2.lons[i]) / 2
mid_lat = (w1.lats[i] + w2.lats[i]) / 2

gtxPath, kmlPath = getJinjaPaths()
jinjaEnv = getJinjaEnv(gtxPath, kmlPath)

gtxFile = os.path.join(gtxPath, 'test.gtx')
saveGtx(gtxFile, jinjaEnv, mid_lon, mid_lat)

kmlFile = os.path.join(kmlPath, 'test.kml')
saveKml(kmlFile, jinjaEnv, mid_lon, mid_lat, 'Weymouth Speed Week')

In [13]:
print("All Done!")

All Done!
